In [1]:
# ===================== BLOCK 0: Install (pin OpenCV 4.x, no pandas upgrade) =====================
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip -q install "opencv-python<5" "opencv-python-headless<5" ultralytics roboflow onnx onnxruntime onnxslim psutil pyyaml
!pip -q install --force-reinstall --no-deps "opencv-python-headless<5"   # leave ONE consistent set of cv2 files
# >>> Now: Runtime -> Restart session, then run Block 1 onward (skip this cell). <<<

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 kB 5.5 MB/s eta 0:00:00


In [2]:
# ===================== BLOCK 1: Raspberry Pi 4B (4 GB) simulated environment =====================
import os, sys, platform

PI = {
    "name": "Raspberry Pi 4 Model B (4 GB)",
    "soc": "BCM2711, 4x Cortex-A72 @ 1.5 GHz",
    "cores": 4,
    "ram_mb": 4096,
    "os_overhead_mb": 500,
    "gpu_inference": False,
    # P = idle + (load - idle) * cpu_utilisation  (verify and cite primary sources in the paper)
    "power_scenarios_w": {
        "low":     {"idle": 2.85, "load": 6.18},
        "typical": {"idle": 3.17, "load": 6.18},
        "high":    {"idle": 3.17, "load": 8.00},
    },
}

N_THREADS = min(PI["cores"], os.cpu_count() or 1)

os.environ["CUDA_VISIBLE_DEVICES"] = ""
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = str(N_THREADS)
os.environ["YOLO_VERBOSE"] = "False"

try:
    os.sched_setaffinity(0, set(range(N_THREADS)))
except Exception as e:
    print("Affinity not set:", e)

if "torch" in sys.modules or "cv2" in sys.modules:
    print("WARNING: torch/cv2 already imported. Restart the runtime and run this block first.")
if N_THREADS < PI["cores"]:
    print(f"NOTE: host exposes only {N_THREADS} cores (Pi has {PI['cores']}); the proxy runs on {N_THREADS} threads. "
          "State this limitation in the paper (the anchor calibration in Block 7 absorbs part of it).")

print(f"Simulated device : {PI['name']} | {PI['soc']}")
print(f"Host CPU cores   : {os.cpu_count()}  -> threads used: {N_THREADS}")
print(f"RAM budget       : {PI['ram_mb']} MB (OS overhead assumed {PI['os_overhead_mb']} MB)")
print(f"Python           : {platform.python_version()} | GPU hidden: True")

NOTE: host exposes only 2 cores (Pi has 4); the proxy runs on 2 threads. State this limitation in the paper (the anchor calibration in Block 7 absorbs part of it).
Simulated device : Raspberry Pi 4 Model B (4 GB) | BCM2711, 4x Cortex-A72 @ 1.5 GHz
Host CPU cores   : 2  -> threads used: 2
RAM budget       : 4096 MB (OS overhead assumed 500 MB)
Python           : 3.13.16 | GPU hidden: True


In [3]:
# ===================== BLOCK 3: Drive, paths, model files =====================
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR  = "/content/drive/MyDrive/moveme_updated_paper_NE/model_files"
DARKNET_CFG_NAME = "custom-yolov4-tiny-detector.cfg"

# family: darknet | yolov5 (original repo checkpoint) | ultralytics
MODELS = {
    "YOLOv4-tiny": {"file": "best_yolov4.weights", "family": "darknet"},
    "YOLOv5":      {"file": "best_yolov5.pt",      "family": "yolov5"},
    "YOLOv8":      {"file": "best_yolov8.pt",      "family": "ultralytics"},
    "YOLO11":      {"file": "best_yolov11.pt",     "family": "ultralytics"},
    "YOLO12":      {"file": "best_yolov12.pt",     "family": "ultralytics"},
    "YOLO26":      {"file": "best_yolo26.pt",      "family": "ultralytics"},
}

WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

LOCAL = {}
for name, info in MODELS.items():
    src = os.path.join(MODEL_DIR, info["file"])
    dst = os.path.join(WORK, info["file"])
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        LOCAL[name] = dst
        print(f"OK      {name:12s} {os.path.getsize(dst)/1e6:7.1f} MB")
    else:
        print(f"MISSING {name:12s} {src}")

cfg_src = os.path.join(MODEL_DIR, DARKNET_CFG_NAME)
DARKNET_CFG_LOCAL = os.path.join(WORK, DARKNET_CFG_NAME)
if os.path.exists(cfg_src):
    shutil.copy(cfg_src, DARKNET_CFG_LOCAL)
    print("OK      darknet cfg")
else:
    print("MISSING darknet cfg -> YOLOv4-tiny will be skipped. Put the .cfg in MODEL_DIR.")
    LOCAL.pop("YOLOv4-tiny", None)

Mounted at /content/drive
OK      YOLOv4-tiny     23.6 MB
OK      YOLOv5          14.4 MB
OK      YOLOv8          22.5 MB
OK      YOLO11          19.2 MB
OK      YOLO12          18.6 MB
OK      YOLO26          44.0 MB
OK      darknet cfg


In [4]:
# ===================== BLOCK 4: Roboflow dataset + test images =====================
import os, glob, yaml
from roboflow import Roboflow

ROBOFLOW_API_KEY = "Gl8JhnrZ9V10NSuIm2PR"   # <-- paste your key between the quotes

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("multilayervehicle").project("back-camera-new-gm1lk")
version = project.version(3)
dataset = version.download("yolov8", location="/content/dataset")

DATA_ROOT = dataset.location
with open(os.path.join(DATA_ROOT, "data.yaml")) as f:
    data_cfg = yaml.safe_load(f)
print("Classes:", data_cfg.get("names"))

MAX_TEST_IMAGES = None
TEST_IMAGES = sorted(glob.glob(os.path.join(DATA_ROOT, "test", "images", "*.*")))
TEST_IMAGES = [p for p in TEST_IMAGES if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
if MAX_TEST_IMAGES:
    TEST_IMAGES = TEST_IMAGES[:MAX_TEST_IMAGES]
print("Test images:", len(TEST_IMAGES))

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/dataset in yolov8:: 100%|██████████| 3361/3361 [00:00<00:00, 6724.22it/s]


Classes: ['2-wheelers', 'bus', 'cargo van', 'jeep', 'minibus', 'person', 'sedan', 'suv', 'tricycle', 'truck', 'utility vehicle', 'van']
Test images: 71


In [5]:
# ===================== BLOCK 5: Helpers + complexity (params, GFLOPs, size) + ONNX export =====================
import os, gc, sys, json, subprocess
import numpy as np, pandas as pd, cv2
from IPython.display import display

cv2.setNumThreads(N_THREADS)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
IMGSZ = 640

# ---- OpenCV sanity (the root cause of the original crash) ----
DARKNET_OK = hasattr(cv2.dnn, "readNetFromDarknet") and int(cv2.__version__.split(".")[0]) < 5
print("OpenCV", cv2.__version__, "| Darknet support:", DARKNET_OK)
if not DARKNET_OK and "YOLOv4-tiny" in LOCAL:
    print("!! OpenCV >= 5 has no readNetFromDarknet. Run Block 0, restart the runtime. YOLOv4-tiny skipped for now.")
    LOCAL.pop("YOLOv4-tiny")

# ---- shared pre/post-processing (identical NMS for every non-Ultralytics model) ----
def letterbox(im, size, color=114):
    h, w = im.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    if (nh, nw) != (h, w):
        im = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, left = (size - nh) // 2, (size - nw) // 2
    out = np.full((size, size, 3), color, dtype=im.dtype)
    out[top:top + nh, left:left + nw] = im
    return out

def nms_np(xyxy, scores, cls, iou_thr, max_det=300):
    """Class-aware NMS (same behaviour as Ultralytics' default)."""
    if len(scores) == 0:
        return []
    b = xyxy + cls[:, None].astype(np.float32) * 8192.0
    x1, y1, x2, y2 = b.T
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size and len(keep) < max_det:
        i = order[0]; keep.append(int(i))
        rest = order[1:]
        if not rest.size:
            break
        iw = np.clip(np.minimum(x2[i], x2[rest]) - np.maximum(x1[i], x1[rest]), 0, None)
        ih = np.clip(np.minimum(y2[i], y2[rest]) - np.maximum(y1[i], y1[rest]), 0, None)
        inter = iw * ih
        iou = inter / (areas[i] + areas[rest] - inter + 1e-9)
        order = rest[iou <= iou_thr]
    return keep

# ---- Darknet: analytic params/GFLOPs from the .cfg (no OpenCV needed, no silent NaN) ----
def parse_darknet_cfg(cfg_path):
    blocks, cur = [], None
    with open(cfg_path) as f:
        for line in f:
            line = line.split("#")[0].strip()
            if not line:
                continue
            if line.startswith("["):
                cur = {"type": line.strip("[]").strip()}; blocks.append(cur)
            elif "=" in line and cur is not None:
                k, v = line.split("=", 1); cur[k.strip()] = v.strip()
    return blocks

def darknet_stats(cfg_path, size):
    """params = conv weights + folded bias (comparable to Ultralytics' FUSED count); GFLOPs = 2 x MACs."""
    blocks = parse_darknet_cfg(cfg_path)
    c, h, w = int(blocks[0].get("channels", 3)), size, size
    shapes, params, macs, floats = [], 0, 0, 0
    for i, b in enumerate(blocks[1:]):
        t = b["type"]
        if t == "convolutional":
            f, k, s = int(b["filters"]), int(b["size"]), int(b["stride"])
            g, bn = int(b.get("groups", 1)), int(b.get("batch_normalize", 0))
            pad = k // 2 if int(b.get("pad", 0)) else int(b.get("padding", 0))
            h, w = (h + 2 * pad - k) // s + 1, (w + 2 * pad - k) // s + 1
            nw = f * (c // g) * k * k
            params += nw + f; macs += nw * h * w
            floats += nw + f + (3 * f if bn else 0)
            c = f
        elif t == "route":
            idx = [int(x) for x in b["layers"].split(",")]
            idx = [i + j if j < 0 else j for j in idx]
            c = sum(shapes[j][0] for j in idx) // int(b.get("groups", 1))
            h, w = shapes[idx[0]][1:]
        elif t == "maxpool":
            k, s = int(b["size"]), int(b.get("stride", 1))
            pad = int(b.get("padding", k - 1))
            h, w = (h + pad - k) // s + 1, (w + pad - k) // s + 1
        elif t == "upsample":
            s = int(b.get("stride", 2)); h, w = h * s, w * s
        elif t in ("shortcut", "yolo"):
            pass
        else:
            raise ValueError(f"unsupported darknet layer: {t}")
        shapes.append((c, h, w))
    return params / 1e6, 2 * macs / 1e9, floats

def darknet_complexity(weights, cfg, size):
    params, gflops, floats = darknet_stats(cfg, size)
    major, minor, _ = np.fromfile(weights, dtype=np.int32, count=3)
    expected = 12 + (8 if major * 10 + minor >= 2 else 4) + 4 * floats
    if expected != os.path.getsize(weights):
        print(f"!! WARNING: cfg expects {expected} bytes but weights file has {os.path.getsize(weights)} "
              "-> this .cfg is NOT the one used to train these weights.")
    return params, gflops

# ---- YOLO12: compatibility shim for checkpoints trained with the original sunsmarterjie/yolov12 fork ----
def apply_yolo12_compat():
    from ultralytics.nn.modules import block as _blk
    AAttn = _blk.AAttn
    if getattr(AAttn, "_compat_patched", False):
        return
    _orig = AAttn.forward
    def forward(self, x):
        if hasattr(self, "qkv"):                 # official layout
            return _orig(self, x)
        B, C, H, W = x.shape                     # original-fork layout (qk + v), non-flash path
        N = H * W
        qk = self.qk(x).flatten(2).transpose(1, 2)
        v = self.v(x)
        pp = self.pe(v)
        v = v.flatten(2).transpose(1, 2)
        if self.area > 1:
            qk = qk.reshape(B * self.area, N // self.area, C * 2)
            v = v.reshape(B * self.area, N // self.area, C)
            B, N, _ = qk.shape
        q, k = qk.split([C, C], dim=2)
        q = q.transpose(1, 2).reshape(B, self.num_heads, self.head_dim, N)
        k = k.transpose(1, 2).reshape(B, self.num_heads, self.head_dim, N)
        v = v.transpose(1, 2).reshape(B, self.num_heads, self.head_dim, N)
        attn = ((q.transpose(-2, -1) @ k) * (self.head_dim ** -0.5)).softmax(dim=-1)
        x = (v @ attn.transpose(-2, -1)).permute(0, 3, 1, 2)
        if self.area > 1:
            x = x.reshape(B // self.area, N * self.area, C)
            B, N, _ = x.shape
        x = x.reshape(B, H, W, C).permute(0, 3, 1, 2)
        return self.proj(x + pp)
    AAttn.forward = forward
    AAttn._compat_patched = True

from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops
apply_yolo12_compat()

def ultra_complexity(path):
    m = YOLO(path)
    try:
        sc = getattr(m.model, "yaml", {}).get("scale")
        print(f"   [{os.path.basename(path)}] trained with ultralytics {m.ckpt.get('version')} | scale={sc}")
    except Exception:
        pass
    try:
        m.fuse()                                   # fused counts = what is deployed (matches Darknet fused count)
    except Exception as e:
        print("   fuse failed:", e)
    params = sum(p.numel() for p in m.model.parameters()) / 1e6
    g = float(get_flops(m.model, IMGSZ))          # returns 0.0 on failure -> treat as NaN
    return params, (g if g > 0 else np.nan)

# ---- YOLOv5 (original repo checkpoint): clone repo, export to ONNX, count params/GFLOPs in a subprocess ----
YOLOV5_DIR = "/content/yolov5"
_V5_SCRIPT = r'''
import sys, json, torch
sys.path.insert(0, ".")
w, imgsz = sys.argv[1], int(sys.argv[2])
ck = torch.load(w, map_location="cpu", weights_only=False)
model = (ck.get("ema") or ck["model"]).float().eval()
model.fuse()
params = sum(p.numel() for p in model.parameters())
try:
    from ultralytics.utils.torch_utils import get_flops
    g = float(get_flops(model, imgsz))
except Exception:
    g = float("nan")
print("RESULT " + json.dumps({"params": params, "gflops": g}))
'''
def yolov5_prepare(weights):
    if not os.path.isdir(YOLOV5_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ultralytics/yolov5", YOLOV5_DIR], check=True)
    env = dict(os.environ, TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1", CUDA_VISIBLE_DEVICES="")
    r = subprocess.run([sys.executable, "-c", _V5_SCRIPT, weights, str(IMGSZ)], cwd=YOLOV5_DIR,
                       env=env, capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
    if not line:
        raise RuntimeError("YOLOv5 stats failed:\n" + r.stderr[-1500:])
    st = json.loads(line[0][7:])
    r = subprocess.run([sys.executable, "export.py", "--weights", weights, "--include", "onnx",
                        "--imgsz", str(IMGSZ), "--device", "cpu"], cwd=YOLOV5_DIR, env=env,
                       capture_output=True, text=True)
    onnx_path = os.path.splitext(weights)[0] + ".onnx"
    if not os.path.exists(onnx_path):
        raise RuntimeError("YOLOv5 ONNX export failed:\n" + (r.stdout + r.stderr)[-1500:])
    g = st["gflops"]
    return st["params"] / 1e6, (g if g and g > 0 else np.nan), onnx_path

# ---- build Table 1 ----
RUNTIME = {}      # model -> (path used for inference, runtime label, kind)
rows = []
for name, info in MODELS.items():
    if name not in LOCAL:
        continue
    try:
        if info["family"] == "darknet":
            p, g = darknet_complexity(LOCAL[name], DARKNET_CFG_LOCAL, IMGSZ)
            RUNTIME[name] = (LOCAL[name], "OpenCV DNN (CPU)", "darknet")
        elif info["family"] == "yolov5":
            p, g, onnx_path = yolov5_prepare(LOCAL[name])
            RUNTIME[name] = (onnx_path, "ONNX Runtime (CPU)", "yolov5")
        else:
            p, g = ultra_complexity(LOCAL[name])
            try:
                onnx_path = YOLO(LOCAL[name]).export(format="onnx", imgsz=IMGSZ, simplify=False,
                                                     dynamic=False, half=False, device="cpu")
                RUNTIME[name] = (onnx_path, "ONNX Runtime (CPU)", "ultra")
            except Exception as e:
                print(f"[{name}] ONNX export failed ({e}) -> PyTorch CPU fallback")
                RUNTIME[name] = (LOCAL[name], "PyTorch (CPU)", "ultra")
    except Exception as e:
        print(f"[{name}] SKIPPED: {e}")
        RUNTIME.pop(name, None)
        continue
    rows.append({"model": name, "runtime": RUNTIME[name][1], "params_M": p,
                 "GFLOPs": g, "weights_MB": os.path.getsize(LOCAL[name]) / 1e6})
    gc.collect()

df_complexity = pd.DataFrame(rows)
print(f"\nTABLE 1: Model complexity at {IMGSZ}x{IMGSZ} (fused parameters; GFLOPs = 2 x MACs)")
display(df_complexity.round(2))

OpenCV 4.14.0 | Darknet support: True
   [best_yolov8.pt] trained with ultralytics 8.4.175 | scale=None
   [best_yolov11.pt] trained with ultralytics 8.3.40 | scale=s
   [best_yolov12.pt] trained with ultralytics 8.3.63 | scale=s
   [best_yolo26.pt] trained with ultralytics 8.4.175 | scale=m

TABLE 1: Model complexity at 640x640 (fused parameters; GFLOPs = 2 x MACs)


,model,runtime,params_M,GFLOPs,weights_MB
0,YOLOv4-tiny,OpenCV DNN (CPU),5.90,16.10,23.62
1,YOLOv5,ONNX Runtime (CPU),7.04,15.85,14.36
2,YOLOv8,ONNX Runtime (CPU),11.13,28.46,22.51
3,YOLO11,ONNX Runtime (CPU),9.42,21.44,19.17
4,YOLO12,ONNX Runtime (CPU),9.08,21.25,18.63
5,YOLO26,ONNX Runtime (CPU),20.36,68.13,44.04


In [6]:
# ===================== BLOCK 6: Inference benchmark on the test set =====================
import time, threading, psutil
import onnxruntime as ort
import torch
torch.set_num_threads(N_THREADS)

CONF, IOU = 0.25, 0.45
imgs = [cv2.imread(p) for p in TEST_IMAGES]
imgs = [i for i in imgs if i is not None]
print(len(imgs), "test images loaded in memory")

def _summ(x):
    x = np.asarray(x, dtype=float)
    return float(np.median(x)), float(np.mean(x)), float(np.percentile(x, 95))

class PeakRSS:
    def __init__(self, interval=0.02):
        self.p = psutil.Process(); self.interval = interval
        self.peak = 0; self._stop = threading.Event()
    def _run(self):
        while not self._stop.is_set():
            self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(self.interval)
    def __enter__(self):
        self.peak = self.p.memory_info().rss
        self.t = threading.Thread(target=self._run, daemon=True); self.t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self.t.join()

def _loop(one, images, warmup):
    for im in images[:warmup]:
        one(im)
    pre, inf, post, e2e, nd = [], [], [], [], []
    c0, w0 = time.process_time(), time.perf_counter()
    for im in images:
        t = time.perf_counter()
        a, b, c, n = one(im)
        e2e.append((time.perf_counter() - t) * 1000)
        pre.append(a); inf.append(b); post.append(c); nd.append(n)
    return dict(pre=pre, inf=inf, post=post, e2e=e2e, nd=nd,
                cpu=time.process_time() - c0, wall=time.perf_counter() - w0)

# ---------- YOLOv4-tiny (Darknet weights via OpenCV DNN) ----------
def bench_darknet(images, size, warmup=5):
    net = cv2.dnn.readNetFromDarknet(DARKNET_CFG_LOCAL, LOCAL["YOLOv4-tiny"])
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    outs = net.getUnconnectedOutLayersNames()
    def one(im):
        t0 = time.perf_counter()
        blob = cv2.dnn.blobFromImage(im, 1/255.0, (size, size), swapRB=True, crop=False)
        net.setInput(blob)
        t1 = time.perf_counter()
        outputs = net.forward(outs)
        t2 = time.perf_counter()
        if not isinstance(outputs, (list, tuple)):
            outputs = [outputs]
        h, w = im.shape[:2]
        dets = np.vstack([o.reshape(-1, o.shape[-1]) for o in outputs])
        cls_scores = dets[:, 5:]
        cls_ids = cls_scores.argmax(1)
        conf = dets[:, 4] * cls_scores[np.arange(len(dets)), cls_ids]
        m = conf >= CONF
        dets, conf, cls_ids = dets[m], conf[m], cls_ids[m]
        n = 0
        if len(dets):
            xyxy = np.stack([(dets[:, 0] - dets[:, 2] / 2) * w, (dets[:, 1] - dets[:, 3] / 2) * h,
                             (dets[:, 0] + dets[:, 2] / 2) * w, (dets[:, 1] + dets[:, 3] / 2) * h],
                            1).astype(np.float32)
            n = len(nms_np(xyxy, conf.astype(np.float32), cls_ids, IOU))   # class-aware, like Ultralytics
        t3 = time.perf_counter()
        return (t1 - t0) * 1000, (t2 - t1) * 1000, (t3 - t2) * 1000, n
    return _loop(one, images, warmup)

# ---------- YOLOv5 (original-repo ONNX, raw decoded output [1, 25200, 5+nc]) ----------
def bench_yolov5_onnx(path, images, warmup=5):
    so = ort.SessionOptions(); so.intra_op_num_threads = N_THREADS
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0].name
    def one(im):
        t0 = time.perf_counter()
        img = letterbox(im, IMGSZ)
        blob = np.ascontiguousarray(img[:, :, ::-1].transpose(2, 0, 1))[None].astype(np.float32) / 255.0
        t1 = time.perf_counter()
        out = sess.run(None, {inp: blob})[0][0]
        t2 = time.perf_counter()
        sc = out[:, 5:] * out[:, 4:5]
        cls_ids = sc.argmax(1)
        conf = sc[np.arange(len(out)), cls_ids]
        m = conf >= CONF
        out, conf, cls_ids = out[m], conf[m], cls_ids[m]
        n = 0
        if len(out):
            xyxy = np.stack([out[:, 0] - out[:, 2] / 2, out[:, 1] - out[:, 3] / 2,
                             out[:, 0] + out[:, 2] / 2, out[:, 1] + out[:, 3] / 2], 1).astype(np.float32)
            n = len(nms_np(xyxy, conf.astype(np.float32), cls_ids, IOU))
        t3 = time.perf_counter()
        return (t1 - t0) * 1000, (t2 - t1) * 1000, (t3 - t2) * 1000, n
    return _loop(one, images, warmup)

# ---------- Ultralytics models (ONNX Runtime or PyTorch CPU) ----------
def bench_ultra(path, images, warmup=5):
    model = YOLO(path, task="detect")
    kw = dict(imgsz=IMGSZ, conf=CONF, iou=IOU, device="cpu", verbose=False)
    def one(im):
        r = model.predict(im, **kw)[0]
        return r.speed["preprocess"], r.speed["inference"], r.speed["postprocess"], len(r.boxes)
    return _loop(one, images, warmup)

BENCH = {"darknet": lambda n, im: bench_darknet(im, IMGSZ),
         "yolov5":  lambda n, im: bench_yolov5_onnx(RUNTIME[n][0], im),
         "ultra":   lambda n, im: bench_ultra(RUNTIME[n][0], im)}

# ---------- Run every model (one failure no longer kills the loop) ----------
rows, FAILED = [], {}
for name in MODELS:
    if name not in RUNTIME:
        continue
    gc.collect()
    base = psutil.Process().memory_info().rss
    print(f"Running {name} ...", end=" ")
    try:
        with PeakRSS() as pk:
            s = BENCH[RUNTIME[name][2]](name, imgs)
    except Exception as e:
        FAILED[name] = repr(e); print("FAILED:", e); continue
    peak_mb, delta_mb = pk.peak / 1024**2, (pk.peak - base) / 1024**2
    util = min(1.0, (s["cpu"] / s["wall"]) / N_THREADS)
    row = {"model": name, "runtime": RUNTIME[name][1], "n_images": len(imgs)}
    for k in ("pre", "inf", "post", "e2e"):
        med, mean, p95 = _summ(s[k])
        row[f"{k}_med_ms"] = med
        if k == "e2e":
            row["e2e_mean_ms"], row["e2e_p95_ms"] = mean, p95
    row.update(avg_dets=float(np.mean(s["nd"])), cpu_util=util,
               mem_delta_mb=delta_mb, peak_rss_mb=peak_mb,
               fits_ram=(delta_mb + PI["os_overhead_mb"]) < PI["ram_mb"])
    rows.append(row)
    print(f"median {row['e2e_med_ms']:.1f} ms")
    gc.collect()

df_proxy = pd.DataFrame(rows)
print("\nTABLE 2: Measured on the Pi-constrained CPU proxy (host, not yet Pi-scaled)")
display(df_proxy.round(2))
if FAILED:
    print("Failed models:", FAILED)

71 test images loaded in memory
Running YOLOv4-tiny ... median 305.7 ms
Running YOLOv5 ... median 329.0 ms
Running YOLOv8 ... median 583.6 ms
Running YOLO11 ... median 501.4 ms
Running YOLO12 ... median 543.8 ms
Running YOLO26 ... median 1415.3 ms

TABLE 2: Measured on the Pi-constrained CPU proxy (host, not yet Pi-scaled)


,model,runtime,n_images,pre_med_ms,inf_med_ms,post_med_ms,e2e_med_ms,e2e_mean_ms,e2e_p95_ms,avg_dets,cpu_util,mem_delta_mb,peak_rss_mb,fits_ram
0,YOLOv4-tiny,OpenCV DNN (CPU),71,6.73,298.08,1.06,305.69,372.77,550.55,2.49,0.71,37.18,894.14,True
1,YOLOv5,ONNX Runtime (CPU),71,5.90,319.67,3.26,328.97,403.48,649.98,2.76,0.77,8.08,870.36,True
2,YOLOv8,ONNX Runtime (CPU),71,6.24,574.52,1.39,583.62,652.09,906.90,2.68,0.50,35.38,905.80,True
3,YOLO11,ONNX Runtime (CPU),71,6.39,491.90,1.44,501.43,559.15,757.04,2.79,0.50,66.43,972.23,True
4,YOLO12,ONNX Runtime (CPU),71,6.29,534.79,1.46,543.82,605.33,828.03,2.61,0.50,0.03,905.84,True
5,YOLO26,ONNX Runtime (CPU),71,6.26,1407.25,1.53,1415.31,1576.77,2176.13,2.86,0.50,217.41,1123.25,True


In [11]:
# ===================== BLOCK 7: Calibrate to Pi 4 and estimate Pi latency =====================
# Anchor: published YOLOv4-tiny @416x416 on a Pi 4 = 3.4 FPS (Q-engineering ncnn table; verify and cite).
PI_ANCHOR_FPS = 3.4
ANCHOR_CLOCK_FACTOR = 1.0     # ~1.3 if that figure was for a 1.95 GHz overclocked board and you want stock 1.5 GHz
MANUAL_SCALE = None           # set a number to override the calibration

if MANUAL_SCALE is None:
    if "YOLOv4-tiny" not in set(df_proxy["model"]):
        raise RuntimeError("Anchor needs a successful YOLOv4-tiny run (fix OpenCV via Block 0 + restart) or set MANUAL_SCALE.")
    a = bench_darknet(imgs[:50], 416)
    colab_anchor_ms = _summ(a["e2e"])[0]
    PI_ANCHOR_MS = 1000 / PI_ANCHOR_FPS * ANCHOR_CLOCK_FACTOR
    SCALE = PI_ANCHOR_MS / colab_anchor_ms
    _, anchor_gflops = darknet_complexity(LOCAL["YOLOv4-tiny"], DARKNET_CFG_LOCAL, 416)
    EFF_GFLOPS = anchor_gflops / (PI_ANCHOR_MS / 1000)
    print(f"Anchor: host {colab_anchor_ms:.1f} ms vs Pi {PI_ANCHOR_MS:.1f} ms -> SCALE = {SCALE:.2f}")
    print(f"Effective Pi throughput from anchor: {EFF_GFLOPS:.2f} GFLOP/s")
else:
    SCALE, EFF_GFLOPS = MANUAL_SCALE, np.nan

df_pi = df_proxy.merge(df_complexity[["model", "params_M", "GFLOPs", "weights_MB"]], on="model", how="left")

# Estimate 1 (anchor-scaled): host end-to-end latency x SCALE
df_pi["pi_ms_anchor"] = df_pi["e2e_med_ms"] * SCALE
# Estimate 2 (FLOPs-based sensitivity); NaN where GFLOPs unavailable instead of a bogus 0 ms
df_pi["pi_ms_flops"] = df_pi["GFLOPs"] / EFF_GFLOPS * 1000 + (df_pi["pre_med_ms"] + df_pi["post_med_ms"]) * SCALE
df_pi["pi_fps_anchor"] = 1000 / df_pi["pi_ms_anchor"]
df_pi["pi_fps_flops"]  = 1000 / df_pi["pi_ms_flops"]

print("\nTABLE 3:Raspberry Pi 4B latency (two estimates)")
display(df_pi[["model", "runtime", "pi_ms_anchor", "pi_fps_anchor", "pi_ms_flops", "pi_fps_flops",
               "cpu_util", "mem_delta_mb", "fits_ram"]].round(2))

Anchor: host 252.8 ms vs Pi 294.1 ms -> SCALE = 1.16
Effective Pi throughput from anchor: 23.13 GFLOP/s

TABLE 3:Raspberry Pi 4B latency (two estimates)


,model,runtime,pi_ms_anchor,pi_fps_anchor,pi_ms_flops,pi_fps_flops,cpu_util,mem_delta_mb,fits_ram
0,YOLOv4-tiny,OpenCV DNN (CPU),355.63,2.81,705.20,1.42,0.71,37.18,True
1,YOLOv5,ONNX Runtime (CPU),382.71,2.61,695.95,1.44,0.77,8.08,True
2,YOLOv8,ONNX Runtime (CPU),678.96,1.47,1239.34,0.81,0.50,35.38,True
3,YOLO11,ONNX Runtime (CPU),583.34,1.71,936.33,1.07,0.50,66.43,True
4,YOLO12,ONNX Runtime (CPU),632.65,1.58,927.82,1.08,0.50,0.03,True
5,YOLO26,ONNX Runtime (CPU),1646.51,0.61,2955.15,0.34,0.50,217.41,True


In [12]:
# ===================== BLOCK 8: Energy analysis (tables only) =====================
# energy (mJ) = power (W) x time (ms);   P_active = idle + (load - idle) * cpu_util
TARGET_FPS = [1, 2, 5]

rows = []
for _, r in df_pi.iterrows():
    for sc, pw in PI["power_scenarios_w"].items():
        P_act = pw["idle"] + (pw["load"] - pw["idle"]) * r["cpu_util"]
        rows.append({
            "model": r["model"], "scenario": sc, "P_active_W": P_act,
            "lat_anchor_ms": r["pi_ms_anchor"],
            "E_total_mJ_anchor": P_act * r["pi_ms_anchor"],
            "E_above_idle_mJ_anchor": (P_act - pw["idle"]) * r["pi_ms_anchor"],
            "E_total_mJ_flops": P_act * r["pi_ms_flops"],
            "E_above_idle_mJ_flops": (P_act - pw["idle"]) * r["pi_ms_flops"],
            "inferences_per_Wh_anchor": 3600 / (P_act * r["pi_ms_anchor"] / 1000),
        })
df_e1 = pd.DataFrame(rows)
print("TABLE 4: Energy per inference")
display(df_e1.round(2))

pt = PI["power_scenarios_w"]["typical"]
rows = []
for _, r in df_pi.iterrows():
    lat_s = r["pi_ms_anchor"] / 1000
    P_act = pt["idle"] + (pt["load"] - pt["idle"]) * r["cpu_util"]
    for fps in TARGET_FPS:
        duty = min(1.0, fps * lat_s)
        achieved = min(fps, 1 / lat_s)
        P_avg = pt["idle"] + (P_act - pt["idle"]) * duty
        rows.append({
            "model": r["model"], "target_FPS": fps, "real_time_ok": lat_s <= 1 / fps,
            "achieved_FPS": achieved, "cpu_duty": duty, "avg_power_W": P_avg,
            "J_per_frame": P_avg / achieved,
            "Wh_per_24h": P_avg * 24,
            "detection_Wh_per_24h_above_idle": (P_avg - pt["idle"]) * 24,
        })
df_e2 = pd.DataFrame(rows)
print("\nTABLE 5: Continuous operation (typical power scenario)")
display(df_e2.round(3))

g = df_e1.groupby("model")
df_e3 = pd.DataFrame({
    "E_total_mJ_min": g[["E_total_mJ_anchor", "E_total_mJ_flops"]].min().min(axis=1),
    "E_total_mJ_typical_anchor": df_e1[df_e1.scenario == "typical"].set_index("model")["E_total_mJ_anchor"],
    "E_total_mJ_max": g[["E_total_mJ_anchor", "E_total_mJ_flops"]].max().max(axis=1),
}).reset_index().sort_values("E_total_mJ_typical_anchor")
print("\nTABLE 6: Energy per inference range (mJ), sorted by typical value")
display(df_e3.round(1))

TABLE 4: Energy per inference


,model,scenario,P_active_W,lat_anchor_ms,E_total_mJ_anchor,E_above_idle_mJ_anchor,E_total_mJ_flops,E_above_idle_mJ_flops,inferences_per_Wh_anchor
0,YOLOv4-tiny,low,5.23,355.63,1859.54,846.01,3687.42,1677.61,1935.96
1,YOLOv4-tiny,typical,5.32,355.63,1892.05,764.71,3751.88,1516.40,1902.70
2,YOLOv4-tiny,high,6.62,355.63,2354.43,1227.09,4668.77,2433.30,1529.03
3,YOLOv5,low,5.43,382.71,2077.11,986.39,3777.23,1793.76,1733.18
4,YOLOv5,typical,5.50,382.71,2104.78,891.61,3827.56,1621.39,1710.39
5,YOLOv5,high,6.91,382.71,2643.89,1430.72,4807.93,2601.76,1361.63
6,YOLOv8,low,4.51,678.96,3059.94,1124.90,5585.47,2053.35,1176.49
7,YOLOv8,typical,4.67,678.96,3169.10,1016.80,5784.74,1856.03,1135.97
8,YOLOv8,high,5.57,678.96,3783.92,1631.62,6906.99,2978.28,951.40
9,YOLO11,low,4.51,583.34,2629.59,967.07,4220.79,1552.25,1369.04



TABLE 5: Continuous operation (typical power scenario)


,model,target_FPS,real_time_ok,achieved_FPS,cpu_duty,avg_power_W,J_per_frame,Wh_per_24h,detection_Wh_per_24h_above_idle
0,YOLOv4-tiny,1,True,1.000,0.356,3.935,3.935,94.433,18.353
1,YOLOv4-tiny,2,True,2.000,0.711,4.699,2.350,112.786,36.706
2,YOLOv4-tiny,5,False,2.812,1.000,5.320,1.892,127.688,51.608
3,YOLOv5,1,True,1.000,0.383,4.062,4.062,97.479,21.399
4,YOLOv5,2,True,2.000,0.765,4.953,2.477,118.877,42.797
5,YOLOv5,5,False,2.613,1.000,5.500,2.105,131.994,55.914
6,YOLOv8,1,True,1.000,0.679,4.187,4.187,100.483,24.403
7,YOLOv8,2,False,1.473,1.000,4.668,3.169,112.022,35.942
8,YOLOv8,5,False,1.473,1.000,4.668,3.169,112.022,35.942
9,YOLO11,1,True,1.000,0.583,4.044,4.044,97.059,20.979



TABLE 6: Energy per inference range (mJ), sorted by typical value


,model,E_total_mJ_min,E_total_mJ_typical_anchor,E_total_mJ_max
3,YOLOv4-tiny,1859.5,1892.0,4668.8
4,YOLOv5,2077.1,2104.8,4807.9
0,YOLO11,2629.6,2723.3,5219.6
1,YOLO12,2849.0,2950.9,5166.0
5,YOLOv8,3059.9,3169.1,6907.0
2,YOLO26,7424.1,7688.4,16478.7


In [9]:
# ===================== BLOCK 8.5 (optional): test-set mAP50-95 for Ultralytics .pt models =====================
# Roboflow's data.yaml uses "../test/images", which breaks when Ultralytics resolves it; write absolute paths.
eval_yaml = os.path.join(DATA_ROOT, "data_eval.yaml")
with open(eval_yaml, "w") as f:
    yaml.safe_dump({"path": DATA_ROOT, "train": "train/images", "val": "valid/images",
                    "test": "test/images", "names": data_cfg["names"]}, f)

ACC_AUTO = {}
for name, info in MODELS.items():
    if info["family"] != "ultralytics" or name not in LOCAL:
        continue
    try:
        r = YOLO(LOCAL[name]).val(data=eval_yaml, split="test", imgsz=IMGSZ, device="cpu",
                                  batch=8, workers=0, plots=False, verbose=False)
        ACC_AUTO[name] = float(r.box.map)
        print(f"{name:8s} mAP50-95 = {ACC_AUTO[name]:.4f}")
    except Exception as e:
        print(f"{name}: val failed -> {e}")
# YOLOv4-tiny and YOLOv5 are not evaluated here; enter their mAP50-95 manually in Block 9 (same split/metric).

YOLOv8   mAP50-95 = 0.5766
YOLO11   mAP50-95 = 0.5385
YOLO12   mAP50-95 = 0.3281
YOLO26   mAP50-95 = 0.5542


In [16]:
# ===================== BLOCK 9: Multi-criteria decision with AHP-derived weights =====================
ACCURACY = {m: None for m in MODELS}
ACCURACY.update(globals().get("ACC_AUTO", {}))
ACCURACY["YOLOv4-tiny"] = 0.381    # mAP50-95, same test split as the other models
ACCURACY["YOLOv5"]      = 0.496

# Deployment requirement: define BEFORE looking at results and justify it by the application.
MIN_FPS_REQUIRED = 5

# ---------------------------------------------------------------------------------------------
# WEIGHTS via AHP (Analytic Hierarchy Process; Saaty, 1980).
# For each deployment scenario we state pairwise judgements "criterion A is k times as important as B"
# on Saaty's 1-9 scale (1 = equal, 3 = moderately, 5 = strongly, 7 = very strongly).
# Weights = principal eigenvector of the comparison matrix. The Consistency Ratio (CR) checks that the
# judgements do not contradict each other (CR < 0.10 is the accepted limit).
# Scenario logic:
#   safety_first : a missed vehicle/person is costly -> accuracy dominates
#   balanced     : accuracy leads, but speed and energy both matter
#   energy_first : battery/solar-powered unit -> energy dominates
# Latency and energy are treated as equally important within a scenario because energy per frame is
# latency x power; memory is the least important because every model fits easily in 4 GB.
# ---------------------------------------------------------------------------------------------
CRITERIA = ["acc", "lat", "energy", "mem"]
JUDGEMENTS = {   # (A, B): how many times A is more important than B
    "safety_first": {("acc", "lat"): 3, ("acc", "energy"): 3, ("acc", "mem"): 5,
                     ("lat", "energy"): 1, ("lat", "mem"): 2, ("energy", "mem"): 2},
    "balanced":     {("acc", "lat"): 2, ("acc", "energy"): 2, ("acc", "mem"): 3,
                     ("lat", "energy"): 1, ("lat", "mem"): 2, ("energy", "mem"): 2},
    "energy_first": {("energy", "acc"): 3, ("energy", "lat"): 3, ("energy", "mem"): 5,
                     ("acc", "lat"): 1, ("acc", "mem"): 2, ("lat", "mem"): 2},
}
RI = {3: 0.58, 4: 0.90, 5: 1.12}   # Saaty's random index

def ahp_weights(pairs):
    n = len(CRITERIA)
    A = np.ones((n, n))
    for (a, b), v in pairs.items():
        i, j = CRITERIA.index(a), CRITERIA.index(b)
        A[i, j], A[j, i] = v, 1.0 / v
    vals, vecs = np.linalg.eig(A)
    k = np.argmax(vals.real)
    w = np.abs(vecs[:, k].real); w /= w.sum()
    cr = ((vals[k].real - n) / (n - 1)) / RI[n]
    return dict(zip(CRITERIA, w)), cr

WEIGHTS, CR = {}, {}
for sc, pairs in JUDGEMENTS.items():
    WEIGHTS[sc], CR[sc] = ahp_weights(pairs)
    assert CR[sc] < 0.10, f"AHP judgements for '{sc}' are inconsistent (CR={CR[sc]:.3f}); revise them."
print("AHP weights (report this table in the paper):")
display(pd.DataFrame(WEIGHTS).T.assign(CR=pd.Series(CR)).round(3))

# ---------------------------------------------------------------------------------------------
pt = PI["power_scenarios_w"]["typical"]
d = df_pi.copy()
d["P_active_W"] = pt["idle"] + (pt["load"] - pt["idle"]) * d["cpu_util"]
d["energy_mJ"] = d["P_active_W"] * d["pi_ms_anchor"]
d["accuracy"] = d["model"].map(ACCURACY).astype(float)
d["feasible"] = (d["pi_ms_anchor"] <= 1000 / MIN_FPS_REQUIRED) & d["fits_ram"]
use_acc = d["accuracy"].notna().all()
if use_acc:
    d["acc_per_J"] = d["accuracy"] / (d["energy_mJ"] / 1000)
else:
    print("NOTE: ACCURACY not complete -> accuracy criterion excluded from the scores.")

if d["feasible"].any():
    pool = d.index[d["feasible"]]
else:
    pool = d.index
    print(f"NOTE: no model reaches {MIN_FPS_REQUIRED} FPS on the Pi; "
          "all models are ranked on a relative basis.")

def _norm(s, higher_better):
    s = s.astype(float)
    rng = s.max() - s.min()
    if not rng > 0:
        return pd.Series(1.0, index=s.index)
    n = (s - s.min()) / rng
    return n if higher_better else 1 - n

sub = d.loc[pool]
crit_scores = {"lat": _norm(sub["pi_ms_anchor"], False),
               "energy": _norm(sub["energy_mJ"], False),
               "mem": _norm(sub["mem_delta_mb"], False)}
if use_acc:
    crit_scores["acc"] = _norm(sub["accuracy"], True)

for sc, w in WEIGHTS.items():
    used = [c for c in CRITERIA if c in crit_scores]
    score = sum(w[c] * crit_scores[c] for c in used) / sum(w[c] for c in used)
    d[f"score_{sc}"] = score.reindex(d.index)

cols = ["model", "runtime", "params_M", "GFLOPs", "pi_ms_anchor", "pi_fps_anchor", "energy_mJ",
        "mem_delta_mb", "fits_ram", "feasible"] + (["accuracy", "acc_per_J"] if use_acc else []) + \
       [f"score_{k}" for k in WEIGHTS]
print(f"\nTABLE 7: Combined summary (feasible = >= {MIN_FPS_REQUIRED} FPS on Pi and fits in RAM)")
display(d[cols].round(3).sort_values("score_balanced", ascending=False))

print("\nBest model per scenario:")
for k in WEIGHTS:
    s = d[f"score_{k}"]
    print(f"  {k:13s}: {d.loc[s.idxmax(), 'model'] if s.notna().any() else 'n/a'}")

# ---------------------------------------------------------------------------------------------
# SENSITIVITY: does the winner depend on the exact weights? Draw 10,000 random weight sets
# (Dirichlet, uniform over all possible weightings) and count how often each model wins.
# ---------------------------------------------------------------------------------------------
used = [c for c in CRITERIA if c in crit_scores]
M = np.column_stack([crit_scores[c].values for c in used])
W = np.random.default_rng(0).dirichlet(np.ones(len(used)), size=10000)
winners = (W @ M.T).argmax(axis=1)
wins = pd.Series(sub["model"].values[winners]).value_counts(normalize=True) * 100
print("\nSENSITIVITY: share of 10,000 random weightings each model wins (%):")
print(wins.round(1).to_string())

AHP weights (report this table in the paper):


,acc,lat,energy,mem,CR
safety_first,0.532,0.185,0.185,0.097,0.002
balanced,0.424,0.227,0.227,0.122,0.004
energy_first,0.185,0.185,0.532,0.097,0.002


NOTE: no model reaches 5 FPS on the Pi; all models are ranked on a relative basis.

TABLE 7: Combined summary (feasible = >= 5 FPS on Pi and fits in RAM)


,model,runtime,params_M,GFLOPs,pi_ms_anchor,pi_fps_anchor,energy_mJ,mem_delta_mb,fits_ram,feasible,accuracy,acc_per_J,score_safety_first,score_balanced,score_energy_first
2,YOLOv8,ONNX Runtime (CPU),11.130,28.456,678.959,1.473,3169.105,35.383,True,False,0.577,0.182,0.897,0.873,0.821
1,YOLOv5,ONNX Runtime (CPU),7.042,15.849,382.706,2.613,2104.782,8.078,True,False,0.496,0.236,0.813,0.845,0.913
3,YOLO11,ONNX Runtime (CPU),9.417,21.444,583.341,1.714,2723.327,66.434,True,False,0.539,0.198,0.830,0.825,0.833
0,YOLOv4-tiny,OpenCV DNN (CPU),5.896,16.099,355.626,2.812,1892.045,37.184,True,False,0.381,0.201,0.565,0.646,0.837
4,YOLO12,ONNX Runtime (CPU),9.079,21.249,632.654,1.581,2950.917,0.027,True,False,0.328,0.111,0.394,0.486,0.678
5,YOLO26,ONNX Runtime (CPU),20.359,68.133,1646.506,0.607,7688.445,217.414,True,False,0.554,0.072,0.484,0.385,0.169



Best model per scenario:
  safety_first : YOLOv8
  balanced     : YOLOv8
  energy_first : YOLOv5

SENSITIVITY: share of 10,000 random weightings each model wins (%):
YOLOv5         70.1
YOLOv8         26.6
YOLOv4-tiny     3.1
YOLO12          0.2
